In [95]:
# 自动重载外部文件的更新
import numpy as np
%load_ext autoreload
%autoreload 2

# 添加项目路径至path
import os
import sys
currentPath = os.path.join(os.getcwd(),"machinelearningIntro","机器学习实践/关联规则")
sys.path.append(currentPath)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [96]:
import pandas as pd
import numpy as np
import math

In [97]:
datadir = r"D:\PythonEx\machinelearningIntro\机器学习实践\关联规则\data\ratings.csv"
df = pd.read_csv(datadir)

In [98]:
len(df)

1000209

In [99]:
df_in_200 = df.loc[(df["moveid"] < 500) & (df["userid"] < 500)]
df_in_200

,userid,moveid,rating,timestamp
25,1,48,5,978824351
39,1,150,5,978301777
40,1,1,5,978824268
44,1,260,4,978300760
73,2,434,2,978300174
...,...,...,...,...
73732,499,357,4,976215055
73735,499,360,4,976217029
73736,499,361,4,976211832
73740,499,372,3,976215894


In [100]:
df_in_200.groupby(["userid"]).count()

,moveid,rating,timestamp
userid,,,
1,4,4,4
2,19,19,19
3,3,3,3
4,2,2,2
5,35,35,35
...,...,...,...
495,5,5,5
496,14,14,14
497,4,4,4


In [101]:
df_in_200.groupby(["moveid"]).count()

,userid,rating,timestamp
moveid,,,
1,164,164,164
2,62,62,62
3,40,40,40
4,14,14,14
5,23,23,23
...,...,...,...
495,2,2,2
496,1,1,1
497,51,51,51


In [102]:
df = df_in_200

In [103]:
# 计算物品项集，计算物品总数
distinct_item_list = list(df["moveid"].drop_duplicates().sort_values().reset_index(drop=True))
movie_count = len(distinct_item_list)

In [104]:
df1 = pd.get_dummies(df[['userid','moveid']],columns=["moveid"],prefix="",prefix_sep="",dtype=bool)
df1

,userid,1,2,3,4,5,6,7,8,9,...,490,491,492,493,494,495,496,497,498,499
25,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
39,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
40,1,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
44,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
73,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73732,499,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
73735,499,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
73736,499,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
73740,499,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [105]:
%%time
inverted_table = df1.groupby("userid").agg(sum)

CPU times: total: 46.9 ms
Wall time: 46.9 ms


In [106]:
inverted_table

,1,2,3,4,5,6,7,8,9,10,...,490,491,492,493,494,495,496,497,498,499
userid,,,,,,,,,,,,,,,,,,,,,
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
496,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
497,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [107]:
%%time
inverted_table_arr = np.array(inverted_table)
inverted_table_arr

CPU times: total: 0 ns
Wall time: 0 ns


array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [108]:
inverted_table_bin = []
for i in range(len(inverted_table_arr)):
    num = int("".join([str(j) for j in inverted_table_arr[i]]),2)
    inverted_table_bin.append(num)

In [109]:
# 计算同时喜欢同两样物品的人数的方法(物品共现矩阵用)
def like_the_same_two_items_users_count(inverted_table_bin,item1_index,item2_index):
    movie_number = movie_count
    item1_mask = 1 << (movie_number - item1_index)
    item2_mask = 1<<(movie_number - item2_index)
    and_mask = item1_mask|item2_mask

    counter = 0
    for i in inverted_table_bin:
        if i & and_mask == and_mask:
            counter = counter + 1

    return counter

In [110]:
like_the_same_two_items_users_count(inverted_table_bin, 1, 2)

43

In [111]:
%%time
co_occurrence1 = pd.DataFrame([[like_the_same_two_items_users_count(inverted_table_bin,i,j) for i in range(movie_count)] for j in range(movie_count)],columns=distinct_item_list,index=distinct_item_list)
co_occurrence1


CPU times: total: 5.27 s
Wall time: 5.27 s


,1,2,3,4,5,6,7,8,9,10,...,490,491,492,493,494,495,496,497,498,499
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,164,43,19,8,15,36,25,5,5,...,3,4,9,17,12,19,2,0,28,2
3,0,43,62,10,4,7,18,10,4,3,...,0,2,4,7,10,13,1,0,9,1
4,0,19,10,40,6,11,10,14,1,1,...,4,3,5,6,5,6,1,0,9,1
5,0,8,4,6,14,4,2,4,0,1,...,1,1,2,5,5,1,0,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0,19,13,6,1,6,18,9,0,5,...,0,2,5,4,6,34,0,0,6,3
496,0,2,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,2,0,1,0
497,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
498,0,28,9,9,3,3,7,12,0,1,...,2,3,3,8,4,6,1,0,51,2


In [112]:
%%time
# 计算每个物品共有几人购买
item_selected_list = inverted_table.agg(sum,axis=0)

CPU times: total: 0 ns
Wall time: 0 ns


In [113]:
item_selected_list

1      164
2       62
3       40
4       14
5       23
      ... 
495      2
496      1
497     51
498      5
499      4
Length: 437, dtype: int64

In [114]:
print(item_selected_list["110"])

194


In [115]:
def cal_similarity(i,j):
    if i != j:
        return co_occurrence1[i][j]/math.sqrt(item_selected_list[str(i)]*item_selected_list[str(j)])
    else:
        return 1

In [116]:
%%time
item_similarity_matrix = pd.DataFrame([[cal_similarity(i,j) for i in distinct_item_list] for j in distinct_item_list],columns=distinct_item_list,index=distinct_item_list)

CPU times: total: 2.61 s
Wall time: 2.61 s


In [117]:
item_similarity_matrix

,1,2,3,4,5,6,7,8,9,10,...,490,491,492,493,494,495,496,497,498,499
1,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2,0.0,1.000000,0.863461,0.644902,0.211851,0.229335,0.741677,1.200038,0.211667,0.074836,...,0.109985,0.127000,0.219971,0.450183,0.261364,1.706250,0.254000,0.0,1.590293,0.127000
3,0.0,0.863461,1.000000,0.422577,0.131876,0.133243,0.461690,0.597614,0.210819,0.055902,...,0.000000,0.079057,0.121716,0.230783,0.271163,1.453444,0.158114,0.0,0.636396,0.079057
4,0.0,0.644902,0.422577,1.000000,0.334367,0.353919,0.433555,1.414214,0.089087,0.031497,...,0.308607,0.200446,0.257172,0.334367,0.229175,1.133893,0.267261,0.0,1.075706,0.133631
5,0.0,0.211851,0.131876,0.334367,1.000000,0.100409,0.067651,0.315244,0.000000,0.024574,...,0.060193,0.052129,0.080257,0.217391,0.178800,0.147442,0.000000,0.0,0.279751,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,1.706250,1.453444,1.133893,0.147442,0.510754,2.064742,2.405351,0.000000,0.416667,...,0.000000,0.353553,0.680414,0.589768,0.727607,1.000000,0.000000,0.0,1.897367,1.060660
496,0.0,0.254000,0.158114,0.267261,0.000000,0.000000,0.162221,0.377964,0.333333,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.0,0.447214,0.000000
497,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000
498,0.0,1.590293,0.636396,1.075706,0.279751,0.161515,0.507833,2.028370,0.000000,0.052705,...,0.258199,0.335410,0.258199,0.746004,0.306786,1.897367,0.447214,0.0,1.000000,0.447214


In [118]:
# 向任意用户x推荐物品
def recommend_to_userX(x):
    x_has_brought_item_list = inverted_table.loc[x,]
    x_has_not_brought_item_list = [i for i in distinct_item_list if i not in x_has_brought_item_list]
    item_score = 0
    item_has_not_brought_score_dict = dict()
    for item_not_brought in x_has_not_brought_item_list:
        for item_brought in x_has_brought_item_list:
            # 查询用户既往对该物品的打分，如果为空就设为0
            rating = df.loc[(df["userid"] == x) & (df['moveid'] == item_brought),'rating']
            if len(rating) == 1:
                item_brought_ranking = int(rating)
            else:
                item_brought_ranking = 0
            item_score += item_brought_ranking * (item_similarity_matrix.iloc[item_brought][item_not_brought])
        item_has_not_brought_score_dict[item_not_brought] = item_score

    return item_has_not_brought_score_dict

In [119]:
%%time
item_has_not_brought_score_dict = recommend_to_userX(1)

CPU times: total: 52.8 s
Wall time: 52.8 s


In [122]:
pd.DataFrame(item_has_not_brought_score_dict.values(),index=item_has_not_brought_score_dict.keys()).sort_values(by=0,ascending=False)[:5]

,0
499,4839.539849
498,4836.999847
497,4805.193984
496,4805.193984
495,4800.113979
